# AGENT9

La plus simple implémentation de l'idée de Hutto et Myin: un _contentless_ process interagissant avec un _content-involving_ process. 

* L'environnement a un bit d'état.
* L'agent peut effectuer deux actions.
* L'environnement renvoie deux _outcome_ possibles. 
* Le _content-involving_ process a un seul slot pour mémoriser la dernière interaction.


# L'environnement 

* Si l'environnement est dans l'état 0
    * L'action 0 produit l'_outcome_ 0
    * L'action 1 produit l'_outcome_ 1
* Si l'environnement est dans l'état 1
    * L'action 0 produit l'_outcome_ 1
    * L'action 1 produit l'_outcome_ 0

L'état de l'environnement reste stable pendant 5 cycles puis il change.

In [55]:
class Environment1:
    def __init__(self, state=0):
        self.state = state
        self.counter = 0
    def outcome(self, action):
        # Change the environment's state after 5 steps
        self.counter += 1
        if self.counter > 5:
            self.state = 1 - self.state
            self.counter = 0
        # Retur the outcome depending on the state
        if self.state:
            outcome = action
        else:
            outcome = 1 - action 
        return outcome

# L'agent

In [56]:
class Interaction:
    """An interaction is a tuple (action, outcome) with a valence"""
    def __init__(self, action, outcome, valence):
        self.action = action
        self.outcome = outcome
        self.valence = valence

    def key(self):
        """ The key to find this interaction in the dictinary is the string '<action><outcome>'. """
        return f"{self.action}{self.outcome}"

    def __str__(self):
        """ Print interaction in the form '<action><outcome:<valence>' for debug."""
        return f"{self.action}{self.outcome}:{self.valence}"

    def __eq__(self, other):
        """ Interactions are equal if they have the same key """
        return self.key() == other.key()

Le _content-involving_ process mémorise une interaction.

La méthode `valence()` renvoie la valence de l'interaction si elle existe ou 0 sinon.

In [61]:
class Content:
    def __init__(self):
        """The content contains an interaction"""
        self.interaction = None
    def valence(self):
        """Return the interaction's valence if it exists or 0 if it doesn't"""
        if self.interaction is not None:
            return self.interaction.valence
        return 0

Le _content_ mémorise la dernière interaction enactée

L'agent sélectionne une action en utilisant le content:

* Si le _content_ contient un interaction de valence positive alors il la sélectionne.
* Si le _content_ contient une interaction de valence négative alors il sélectionne une interaction avec une action différente.
* Si le _content_ ne contient pas d'interaction alors il sélectionne l'interaction 00


In [62]:
import pandas as pd

class Agent:
    """Creating our agent"""
    def __init__(self, _interactions, content):
        """ Initialize the dictionary of interactions"""
        self._interactions = {interaction.key(): interaction for interaction in _interactions}
        self._intended_interaction = self._interactions["00"]
        self.memory = 0
        self.action_df = pd.DataFrame({"action": [i.action for i in _interactions if i.outcome == 0]}) # , columns=['action', 'outcome', 'valence'])
        self.content = content

    def select_interaction(self):
        """Select the next action"""
        if self.content.valence() > 0:
            self._intended_interaction = self.content.interaction
        elif self.content.valence() < 0:
            negative_action = self.content.interaction.action
            # Find an interaction with a different action
            self._intended_interaction = next((i for i in self._interactions.values() if i.action != negative_action), None)
        else:
            self._intended_interaction = self._interactions["00"]

    def action(self, _outcome):
        """ Tracing the previous cycle """
        previous_interaction = self._interactions[f"{self._intended_interaction.action}{_outcome}"]
        print(f"Action: {self._intended_interaction.action}, Prediction: {self._intended_interaction.outcome}, Outcome: {_outcome}, " 
              f"Prediction: {self._intended_interaction.outcome == _outcome}, Valence: {previous_interaction.valence})")

        """Memorize the previous interaction in the content"""
        self.content.interaction = previous_interaction

        """ Computing the next interaction to try to enact """
        self.select_interaction()
        
        return self._intended_interaction.action

# On lance l'expérimentation

In [63]:
interactions = [
    Interaction(0, 0, -1),
    Interaction(0, 1, 1),
    Interaction(1, 0,-1),
    Interaction(1, 1, 1),
]

a = Agent(interactions, Content())
e = Environment1(0)

outcome = 0
for i in range(20):
    action = a.action(outcome)
    outcome = e.outcome(action)

Action: 0, Prediction: 0, Outcome: 0, Prediction: True, Valence: -1)
Action: 1, Prediction: 0, Outcome: 0, Prediction: True, Valence: -1)
Action: 0, Prediction: 0, Outcome: 1, Prediction: False, Valence: 1)
Action: 0, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 0, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 0, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 0, Prediction: 1, Outcome: 0, Prediction: False, Valence: -1)
Action: 1, Prediction: 0, Outcome: 1, Prediction: False, Valence: 1)
Action: 1, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 1, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 1, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 1, Prediction: 1, Outcome: 1, Prediction: True, Valence: 1)
Action: 1, Prediction: 1, Outcome: 0, Prediction: False, Valence: -1)
Action: 0, Prediction: 0, Outcome: 1, Prediction: False, Valence: 1)
Action: 0, Prediction: 1, Outcome: 1, P

# Conclusion

Ca fonctionne ! l'agent s'adapte quand l'état de l'environnement change. 

Il faut maintenant imaginer un environnement et un _content-involving process_ plus sophistiqués.